In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

PyTorch version: 2.13.0+cu130
GPU available: True
GPU: NVIDIA GeForce RTX 4090
BF16 supported: True


In [2]:
import vllm

print("vLLM version:", vllm.__version__)

vLLM version: 0.32.0


In [3]:
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.used,memory.total",
     "--format=csv"],
    capture_output=True,
    text=True
)

print(result.stdout)

name, memory.used [MiB], memory.total [MiB]
NVIDIA GeForce RTX 4090, 21862 MiB, 24564 MiB



In [6]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)

response = client.chat.completions.create(
    model="Qwen/Qwen3-8B",
    messages=[
        {"role": "user", "content": "Explain what a KV Cache is in two sentences."}
    ],
    max_tokens=500,
    temperature=0,
)

print(response.choices[0].message.content)
print("\nToken usage:", response.usage)

<think>
Okay, the user is asking for an explanation of a KV Cache in two sentences. Let me start by recalling what a KV Cache is. From what I remember, KV stands for Key-Value, so it's a data structure that stores pairs of keys and values. But wait, in the context of machine learning or large language models, maybe it's different. Oh right, in the context of transformers, especially for models like LLaMA or other large models, a KV Cache is used during inference to store the keys and values of attention layers. 

So the main idea is that during the generation of each token, the model computes attention, and the keys and values from previous tokens are stored in a cache. This allows the model to efficiently reuse these values for subsequent tokens without recomputing them, which saves computation time and resources. 

But I need to make sure I'm not mixing up different concepts. Let me check. Yes, in transformer models, the KV Cache is crucial for efficient decoding. When generating a s

as we can see its spending majority of time thinking - so its time to put enable_thinking to FALSE

In [7]:
response = client.chat.completions.create(
    model="Qwen/Qwen3-8B",
    messages=[
        {"role": "user", "content": "Explain what a KV cache is in two sentences."}
    ],
    max_tokens=150,
    temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Answer:", response.choices[0].message.content)
print("Finish reason:", response.choices[0].finish_reason)
print("Token usage:", response.usage)

Answer: A KV cache, or key-value cache, is a data storage mechanism that stores data in pairs of keys and values for quick retrieval. It is commonly used to improve performance by reducing the need to recompute or fetch data repeatedly.
Finish reason: stop
Token usage: CompletionUsage(completion_tokens=47, prompt_tokens=23, total_tokens=70, completion_tokens_details=None, prompt_tokens_details=None)


In [10]:
import time
from concurrent.futures import ThreadPoolExecutor

MODEL = "Qwen/Qwen3-8B"

prompt = prompt_1000 + "\nSummarise the passage in two sentences."


def send_request(request_id):
    start = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=150,
        temperature=0,
        extra_body={
            "chat_template_kwargs": {"enable_thinking": False}
        },
    )

    latency = time.perf_counter() - start

    return {
        "request_id": request_id,
        "latency": latency,
        "prompt_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "finish_reason": response.choices[0].finish_reason,
    }


def run_experiment(concurrency):
    start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        results = list(executor.map(send_request, range(concurrency)))

    total_time = time.perf_counter() - start
    total_output_tokens = sum(r["output_tokens"] for r in results)
    average_latency = sum(r["latency"] for r in results) / len(results)

    print(f"\nConcurrent requests: {concurrency}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average request latency: {average_latency:.2f} seconds")
    print(f"Total output tokens: {total_output_tokens}")
    print(f"Total throughput: {total_output_tokens / total_time:.2f} tokens/sec")

    for r in results:
        print(
            f"  Request {r['request_id'] + 1}: "
            f"{r['prompt_tokens']} prompt + {r['output_tokens']} output tokens, "
            f"latency {r['latency']:.2f}s, "
            f"finish: {r['finish_reason']}"
        )


run_experiment(concurrency=1)
run_experiment(concurrency=10)


Concurrent requests: 1
Total time: 1.23 seconds
Average request latency: 1.22 seconds
Total output tokens: 48
Total throughput: 39.18 tokens/sec
  Request 1: 1010 prompt + 48 output tokens, latency 1.22s, finish: stop

Concurrent requests: 10
Total time: 2.16 seconds
Average request latency: 2.07 seconds
Total output tokens: 469
Total throughput: 217.40 tokens/sec
  Request 1: 1010 prompt + 48 output tokens, latency 2.03s, finish: stop
  Request 2: 1010 prompt + 48 output tokens, latency 2.05s, finish: stop
  Request 3: 1010 prompt + 46 output tokens, latency 2.03s, finish: stop
  Request 4: 1010 prompt + 45 output tokens, latency 2.07s, finish: stop
  Request 5: 1010 prompt + 46 output tokens, latency 2.05s, finish: stop
  Request 6: 1010 prompt + 46 output tokens, latency 2.05s, finish: stop
  Request 7: 1010 prompt + 46 output tokens, latency 2.02s, finish: stop
  Request 8: 1010 prompt + 48 output tokens, latency 2.14s, finish: stop
  Request 9: 1010 prompt + 48 output tokens, lat

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

sentence = (
    "A KV cache stores keys and values from previous tokens "
    "so the model can reuse them during generation. "
)

prompt_1000 = ""

while True:
    candidate = prompt_1000 + sentence
    token_count = len(tokenizer.encode(candidate, add_special_tokens=False))

    if token_count > 1000:
        break

    prompt_1000 = candidate

print("Prompt text tokens:", len(tokenizer.encode(prompt_1000, add_special_tokens=False)))

/workspace/vllm-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Prompt text tokens: 989


In [11]:
run_experiment(concurrency=10)


Concurrent requests: 10
Total time: 2.15 seconds
Average request latency: 2.09 seconds
Total output tokens: 455
Total throughput: 211.94 tokens/sec
  Request 1: 1010 prompt + 45 output tokens, latency 2.12s, finish: stop
  Request 2: 1010 prompt + 48 output tokens, latency 2.09s, finish: stop
  Request 3: 1010 prompt + 46 output tokens, latency 2.07s, finish: stop
  Request 4: 1010 prompt + 45 output tokens, latency 2.09s, finish: stop
  Request 5: 1010 prompt + 45 output tokens, latency 2.06s, finish: stop
  Request 6: 1010 prompt + 46 output tokens, latency 2.06s, finish: stop
  Request 7: 1010 prompt + 45 output tokens, latency 2.06s, finish: stop
  Request 8: 1010 prompt + 45 output tokens, latency 2.08s, finish: stop
  Request 9: 1010 prompt + 45 output tokens, latency 2.10s, finish: stop
  Request 10: 1010 prompt + 45 output tokens, latency 2.12s, finish: stop


with prefix caching enabled

In [12]:
run_experiment(concurrency=1)


Concurrent requests: 1
Total time: 1.31 seconds
Average request latency: 1.31 seconds
Total output tokens: 48
Total throughput: 36.67 tokens/sec
  Request 1: 1010 prompt + 48 output tokens, latency 1.31s, finish: stop


In [13]:
run_experiment(concurrency=10)


Concurrent requests: 10
Total time: 1.32 seconds
Average request latency: 1.30 seconds
Total output tokens: 480
Total throughput: 363.64 tokens/sec
  Request 1: 1010 prompt + 48 output tokens, latency 1.31s, finish: stop
  Request 2: 1010 prompt + 48 output tokens, latency 1.28s, finish: stop
  Request 3: 1010 prompt + 48 output tokens, latency 1.31s, finish: stop
  Request 4: 1010 prompt + 48 output tokens, latency 1.31s, finish: stop
  Request 5: 1010 prompt + 48 output tokens, latency 1.31s, finish: stop
  Request 6: 1010 prompt + 48 output tokens, latency 1.30s, finish: stop
  Request 7: 1010 prompt + 48 output tokens, latency 1.30s, finish: stop
  Request 8: 1010 prompt + 48 output tokens, latency 1.30s, finish: stop
  Request 9: 1010 prompt + 48 output tokens, latency 1.30s, finish: stop
  Request 10: 1010 prompt + 48 output tokens, latency 1.29s, finish: stop


Concurrency increases aggregate cache demand as active sequences accumulate tokens.
Prefix caching can reduce duplicate cache work and improve performance when requests share a prefix.